# PyTorch ile Derin Öğrenme · Autograd: hesaplama grafiği ve geri yayılım

Macademia · Büyükler için

> **Colab'da çalıştırma:** Bu dosyayı [colab.research.google.com](https://colab.research.google.com) adresinde **Dosya → Not defteri yükle** ile aç. GPU için **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri yukarıdan aşağı sırayla çalıştır (Shift + Enter). PyTorch Colab'da kuruludur; bilgisayarında çalıştırmak istersen `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # tekrarlanabilirlik
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Tek nöron
$z = w_1x_1 + w_2x_2 + b$, $y = \tanh(z)$, $L = (y - t)^2$

In [ ]:
x1, x2, t = torch.tensor(2.0), torch.tensor(0.5), torch.tensor(0.8)
w1 = torch.tensor(-0.5, requires_grad=True)
w2 = torch.tensor(1.2, requires_grad=True)
b = torch.tensor(0.3, requires_grad=True)
y = torch.tanh(w1 * x1 + w2 * x2 + b)
L = (y - t) ** 2
print('y =', y.item(), 'L =', L.item())
print(L.grad_fn, '->', L.grad_fn.next_functions)                     # grafik düğümleri

## 2. Geri yayılım ve elle kontrol

In [ ]:
L.backward()
manual = 2 * (y - t) * (1 - y ** 2) * x1                              # zincir kuralı
print('autograd:', w1.grad.item(), '| elle:', manual.item())

## 3. Sayısal türev
Merkezî fark: $(L(\theta+h) - L(\theta-h)) / 2h$

In [ ]:
def loss(w):
    return ((torch.tanh(w * x1 + w2 * x2 + b) - t) ** 2).item()
h = 1e-4
print('sayısal:', (loss(w1.item() + h) - loss(w1.item() - h)) / (2 * h))
print(torch.autograd.gradcheck(lambda w: torch.tanh(w * 2.0) ** 2, torch.tensor([0.3], dtype=torch.float64, requires_grad=True)))

## 4. Gradyanlar birikir

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (w * 3).backward()
    print(i, w.grad.item())                                          # 3, 6, 9: birikir
w.grad = None

## 5. Elle gradyan inişi

In [ ]:
for step in range(20):
    y = torch.tanh(w1 * x1 + w2 * x2 + b)
    L = (y - t) ** 2
    L.backward()
    with torch.no_grad():                                            # güncelleme grafiğe girmesin
        for p in (w1, w2, b):
            p -= 0.1 * p.grad
            p.grad = None
    if step % 5 == 0:
        print(step, round(L.item(), 5))

## Alıştırmalar
1. $x_1 = 0$ yap: $w_1$'in gradyanı neden sıfır?
2. `detach()` ile $y$'yi grafikten kopar ve `backward` dene: ne oldu?
3. Kendi fonksiyonun için `gradcheck` yaz.